# ENM 5310 Project — AuroraSmallPretrained (Clean Baseline Notebook)

This notebook:
- Loads ERA5 data (static, surface, atmospheric)
- Runs **AuroraSmallPretrained** forward pass
- Builds a **6-hour accumulated precipitation target**
- Computes and saves **normalization statistics**

No training is done yet. This is a **clean, verified baseline**.

## Step 0 — Device Selection

In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: cuda


## Step 1 — Download ERA5 Data

We download:
- Static fields (geopotential, land/sea mask, soil type)
- Surface variables at 6-hourly resolution
- Atmospheric variables on pressure levels

In [2]:
from pathlib import Path
import cdsapi

download_path = Path("./downloads/era5")
download_path.mkdir(parents=True, exist_ok=True)

c = cdsapi.Client()

# Static variables
static_file = download_path / "static.nc"
if not static_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["geopotential", "land_sea_mask", "soil_type"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": "00:00",
            "format": "netcdf",
        },
        str(static_file),
    )

# Surface variables
surf_file = download_path / "2023-01-01-surface-level.nc"
if not surf_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "2m_temperature",
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "mean_sea_level_pressure",
            ],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(surf_file),
    )

# Atmospheric variables
atm_file = download_path / "2023-01-01-atmospheric.nc"
if not atm_file.exists():
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": ["temperature", "u_component_of_wind", "v_component_of_wind", "specific_humidity", "geopotential"],
            "pressure_level": ["50","100","150","200","250","300","400","500","600","700","850","925","1000"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(atm_file),
    )

print("ERA5 data ready")

2025-12-17 14:19:00,309 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.


ERA5 data ready


## Step 2 — Build Aurora Batch

In [3]:
import xarray as xr
from aurora import Batch, Metadata

static_ds = xr.open_dataset(static_file)
surf_ds   = xr.open_dataset(surf_file)
atm_ds    = xr.open_dataset(atm_file)

batch = Batch(
    surf_vars={
        "2t":  torch.from_numpy(surf_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_ds["msl"].values[:2][None]),
    },
    static_vars={
        "z":   torch.from_numpy(static_ds["z"].values[0]),
        "slt": torch.from_numpy(static_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atm_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atm_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atm_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atm_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atm_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_ds.latitude.values),
        lon=torch.from_numpy(surf_ds.longitude.values),
        time=(surf_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(l) for l in atm_ds.pressure_level.values),
    ),
)

## Step 3 — Run AuroraSmallPretrained (Sanity Check)

In [4]:
from aurora import AuroraSmallPretrained, rollout
import torch

model = AuroraSmallPretrained(use_lora=False)
model.load_checkpoint("microsoft/aurora", "aurora-0.25-small-pretrained.ckpt")
model.eval()

model = model.to("cuda")
batch_cuda = batch.to("cuda")

with torch.inference_mode():
    preds = [p.to("cpu") for p in rollout(model, batch_cuda, steps=1)]

# free GPU
del batch_cuda
model = model.to("cpu")
torch.cuda.empty_cache()


## Step 4 — Build Precipitation Target (ERA5)

Target:
- 6-hour accumulated precipitation
- log(1 + meters)

In [5]:
tp_file = download_path / "2023-01-01-tp-hourly-01_to_06.nc"
if not tp_file.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": ["total_precipitation"],
            "year": "2023",
            "month": "01",
            "day": "01",
            "time": [f"{h:02d}:00" for h in range(1, 7)],
            "format": "netcdf",
        },
        str(tp_file),
    )

tp_ds = xr.open_dataset(tp_file)
tp = tp_ds["tp"]

tp6h = tp.sum(dim="valid_time")
y = torch.from_numpy((tp6h.values).astype("float32"))
y_log = torch.log1p(y)

## Step 5 — Compute & Save Normalization Statistics

In [6]:
import json

vals = y_log.numpy().ravel()
vals = vals[~torch.isnan(torch.from_numpy(vals)).numpy()]

mu = float(vals.mean())
std = float(max(vals.std(), 1e-6))

stats = {
    "target": "log1p_tp_6h",
    "mean": mu,
    "std": std,
    "units": "log(1 + meters)",
}

with open("tp_normalization.json", "w") as f:
    json.dump(stats, f, indent=2)

print(stats)

{'target': 'log1p_tp_6h', 'mean': 0.0005497062811627984, 'std': 0.0019855170976370573, 'units': 'log(1 + meters)'}


# step 6

In [7]:
# Step 6.1 — Build a feature tensor from Aurora output
# We use Aurora's predicted surface variables as features for the precipitation head.

def pack_surf_features(pred_batch):
    """
    pred_batch is Aurora output Batch (pred).
    Returns X: [B, C, H, W] float32 on same device as pred_batch tensors.
    """
    # Each is [B, 1, H, W]
    x2t  = pred_batch.surf_vars["2t"]
    x10u = pred_batch.surf_vars["10u"]
    x10v = pred_batch.surf_vars["10v"]
    xmsl = pred_batch.surf_vars["msl"]

    # Concatenate along channel dimension -> [B, 4, H, W]
    X = torch.cat([x2t, x10u, x10v, xmsl], dim=1).float()
    return X


In [11]:
import torch.nn as nn

class PrecipHead(nn.Module):
    def __init__(self, in_ch=4):
        super().__init__()
        # Minimal, readable CNN head
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=1),   # output: [B,1,H,W]
        )

    def forward(self, x):
        return self.net(x)

# Freeze Aurora backbone
for p in model.parameters():
    p.requires_grad_(False)

head = PrecipHead(in_ch=4).to("cpu")



In [ ]:
import json

with open("tp_normalization.json", "r") as f:
    stats = json.load(f)

mu = stats["mean"]
std = stats["std"]

# y_log is [H, W] right now (from your Step 4)
# make it [B,1,H,W]
y_log_bchw = y_log[None, None].to("cpu").float()

# normalize
y_norm = (y_log_bchw - mu) / std
y_norm = y_norm[:, :, :720, :]     # TO MATCH the other one, this was 721 


In [ ]:
model.eval()
head.train()

batch_gpu = batch.to(device)

with torch.inference_mode():
    pred = model(batch_gpu)      # Aurora output Batch

X = pack_surf_features(pred)     # [B,4,H,W]
yhat = head(X)                   # [B,1,H,W]

print("X:", X.shape, X.dtype, X.device)
print("yhat:", yhat.shape, yhat.dtype, yhat.device)
print("y_norm:", y_norm.shape, y_norm.dtype, y_norm.device)


In [13]:

yhat = head(X)                   # [B,1,H,W]

print("X:", X.shape, X.dtype, X.device)
print("yhat:", yhat.shape, yhat.dtype, yhat.device)
print("y_norm:", y_norm.shape, y_norm.dtype, y_norm.device)

X: torch.Size([1, 4, 720, 1440]) torch.float32 cpu
yhat: torch.Size([1, 1, 720, 1440]) torch.float32 cpu
y_norm: torch.Size([1, 1, 721, 1440]) torch.float32 cpu


In [18]:
import torch.optim as optim
loss_fn = nn.MSELoss()
opt = optim.Adam(head.parameters(), lr=1e-3)

model.eval()
head.train()

batch_gpu = batch.to(device)

# Compute features ONCE (Aurora is frozen)
with torch.inference_mode():
    pred = model(batch_gpu)
X = pack_surf_features(pred).detach()   # fixed tensor

for step in range(50):
    opt.zero_grad()
    yhat = head(X)
    loss = loss_fn(yhat, y_norm)
    loss.backward()
    opt.step()

    if step % 10 == 0:
        print(f"step {step:03d} | loss {loss.item():.6f}")


step 000 | loss 21.456913
step 010 | loss 1811332.250000
step 020 | loss 10094.194336
step 030 | loss 204108.671875
step 040 | loss 1674.337036


In [20]:
print(X.device)
print(y_norm.device)
print(next(head.parameters()).device)


cpu
cpu
cpu
